# Phase 1 - Data Transformation

## Name: AYUSH KUMAR YADAV
## Registration Number: 23BDS0333
## Dataset: Lithium-Ion Battery Materials

### Objective
The objective of this notebook is to transform the cleaned Lithium-Ion
Battery Materials dataset into suitable forms for statistical analysis
and exploratory data analysis.

The transformation process includes creating derived variables,
normalizing numerical attributes using Min-Max scaling, standardizing
numerical attributes using Z-score standardization, and applying
logarithmic transformation where appropriate.

The transformed dataset will be saved for use in subsequent analysis.

In [1]:
# ============================================================
# PHASE 1 - DATA TRANSFORMATION
# Name        : AYUSH KUMAR YADAV
# Registration: 23BDS0333
# Dataset     : Lithium-Ion Battery Materials
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ------------------------------------------------------------
# 1. LOAD CLEANED DATASET
# ------------------------------------------------------------

file_path = "lithium_ion_cleaned.csv"

df = pd.read_csv(file_path)

print("=" * 75)
print("DATA TRANSFORMATION")
print("=" * 75)

print("\nORIGINAL DATASET SHAPE:")
print(df.shape)

# ------------------------------------------------------------
# 2. IDENTIFY NUMERICAL VARIABLES
# ------------------------------------------------------------

numerical_columns = df.select_dtypes(include=np.number).columns.tolist()

print("\nNUMERICAL VARIABLES:")
print(numerical_columns)

# ------------------------------------------------------------
# 3. CREATE DERIVED VARIABLES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CREATING DERIVED VARIABLES")
print("=" * 75)

# Energy stability category based on Energy Above Hull
if "e_above_hull_ev" in df.columns:

    df["stability_category"] = pd.cut(
        df["e_above_hull_ev"],
        bins=[-np.inf, 0.05, 0.15, np.inf],
        labels=["Low", "Medium", "High"]
    )

    print("Created: stability_category")

# Density-to-volume relationship
if "density_gm_cc" in df.columns and "volume" in df.columns:

    df["density_volume_index"] = (
        df["density_gm_cc"] / df["volume"]
    )

    print("Created: density_volume_index")

# ------------------------------------------------------------
# 4. MIN-MAX NORMALIZATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MIN-MAX NORMALIZATION")
print("=" * 75)

# Select numerical columns before adding transformed columns
base_numeric_columns = [
    column for column in numerical_columns
    if column in df.columns
]

minmax_scaler = MinMaxScaler()

normalized_data = minmax_scaler.fit_transform(
    df[base_numeric_columns]
)

normalized_df = pd.DataFrame(
    normalized_data,
    columns=[
        column + "_normalized"
        for column in base_numeric_columns
    ],
    index=df.index
)

df = pd.concat([df, normalized_df], axis=1)

print("Min-Max normalized columns created:")
print(normalized_df.columns.tolist())

# ------------------------------------------------------------
# 5. Z-SCORE STANDARDIZATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("Z-SCORE STANDARDIZATION")
print("=" * 75)

standard_scaler = StandardScaler()

standardized_data = standard_scaler.fit_transform(
    df[base_numeric_columns]
)

standardized_df = pd.DataFrame(
    standardized_data,
    columns=[
        column + "_standardized"
        for column in base_numeric_columns
    ],
    index=df.index
)

df = pd.concat([df, standardized_df], axis=1)

print("Standardized columns created:")
print(standardized_df.columns.tolist())

# ------------------------------------------------------------
# 6. LOGARITHMIC TRANSFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("LOGARITHMIC TRANSFORMATION")
print("=" * 75)

# Log transformation is applied only to non-negative
# physical variables.

log_columns = [
    "e_above_hull_ev",
    "band_gap_ev",
    "nsites",
    "density_gm_cc",
    "volume"
]

for column in log_columns:

    if column in df.columns:

        # Values are clipped at zero because log1p
        # requires values >= -1 and these variables
        # represent non-negative physical quantities.
        df[column + "_log"] = np.log1p(
            df[column].clip(lower=0)
        )

        print(f"Created: {column}_log")

# ------------------------------------------------------------
# 7. DISPLAY TRANSFORMED DATA
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("TRANSFORMED DATASET SAMPLE")
print("=" * 75)

display(df.head())

# ------------------------------------------------------------
# 8. CHECK TRANSFORMED VARIABLES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("TRANSFORMED COLUMN NAMES")
print("=" * 75)

for column in df.columns:
    print(column)

# ------------------------------------------------------------
# 9. CHECK NORMALIZED VALUES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MIN-MAX NORMALIZATION CHECK")
print("=" * 75)

normalization_check = df[
    [column + "_normalized" for column in base_numeric_columns]
].agg(["min", "max"]).T

display(normalization_check)

# ------------------------------------------------------------
# 10. CHECK STANDARDIZED VALUES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STANDARDIZATION CHECK")
print("=" * 75)

standardization_check = df[
    [column + "_standardized" for column in base_numeric_columns]
].agg(["mean", "std"]).T

display(standardization_check)

# ------------------------------------------------------------
# 11. CHECK MISSING VALUES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MISSING VALUE CHECK")
print("=" * 75)

print("Total missing values:", df.isnull().sum().sum())

# ------------------------------------------------------------
# 12. SAVE TRANSFORMED DATASET
# ------------------------------------------------------------

output_file = "lithium_ion_transformed.csv"

df.to_csv(output_file, index=False)

print("\n" + "=" * 75)
print("TRANSFORMED DATASET SAVED")
print("=" * 75)

print("Output file:", output_file)
print("Final shape:", df.shape)

print("\n" + "=" * 75)
print("DATA TRANSFORMATION COMPLETED")
print("=" * 75)

DATA TRANSFORMATION

ORIGINAL DATASET SHAPE:
(339, 11)

NUMERICAL VARIABLES:
['formation_energy_ev', 'e_above_hull_ev', 'band_gap_ev', 'nsites', 'density_gm_cc', 'volume']

CREATING DERIVED VARIABLES
Created: stability_category
Created: density_volume_index

MIN-MAX NORMALIZATION
Min-Max normalized columns created:
['formation_energy_ev_normalized', 'e_above_hull_ev_normalized', 'band_gap_ev_normalized', 'nsites_normalized', 'density_gm_cc_normalized', 'volume_normalized']

Z-SCORE STANDARDIZATION
Standardized columns created:
['formation_energy_ev_standardized', 'e_above_hull_ev_standardized', 'band_gap_ev_standardized', 'nsites_standardized', 'density_gm_cc_standardized', 'volume_standardized']

LOGARITHMIC TRANSFORMATION
Created: e_above_hull_ev_log
Created: band_gap_ev_log
Created: nsites_log
Created: density_gm_cc_log
Created: volume_log

TRANSFORMED DATASET SAMPLE


,materials_id,formula,spacegroup,formation_energy_ev,e_above_hull_ev,band_gap_ev,nsites,density_gm_cc,volume,has_bandstructure,...,e_above_hull_ev_standardized,band_gap_ev_standardized,nsites_standardized,density_gm_cc_standardized,volume_standardized,e_above_hull_ev_log,band_gap_ev_log,nsites_log,density_gm_cc_log,volume_log
0,mp-849394,Li2MnSiO4,Pc,-2.699,0.006,3.462,16,2.993,178.513,True,...,-1.718857,1.269778,-0.988690,0.025455,-0.989769,0.005982,1.495597,2.833213,1.384543,5.190248
1,mp-783909,Li2MnSiO4,P21/c,-2.696,0.008,2.879,32,2.926,365.272,True,...,-1.652950,0.732792,-0.296020,-0.164107,-0.350714,0.007968,1.355577,3.496508,1.367621,5.903376
2,mp-761311,Li4MnSi2O7,Cc,-2.775,0.012,3.653,28,2.761,301.775,True,...,-1.521135,1.445704,-0.469187,-0.630939,-0.567989,0.011929,1.537512,3.367296,1.324685,5.712990
3,mp-761598,Li4Mn2Si3O10,C2/c,-2.783,0.013,3.015,38,2.908,436.183,True,...,-1.488181,0.858058,-0.036268,-0.215034,-0.108070,0.012916,1.390037,3.663562,1.363026,6.080352
4,mp-767709,Li2Mn3Si3O10,C2/c,-2.747,0.016,2.578,36,3.334,421.286,True,...,-1.389320,0.455548,-0.122852,0.990242,-0.159045,0.015873,1.274804,3.610918,1.466491,6.045683



TRANSFORMED COLUMN NAMES
materials_id
formula
spacegroup
formation_energy_ev
e_above_hull_ev
band_gap_ev
nsites
density_gm_cc
volume
has_bandstructure
crystal_system
stability_category
density_volume_index
formation_energy_ev_normalized
e_above_hull_ev_normalized
band_gap_ev_normalized
nsites_normalized
density_gm_cc_normalized
volume_normalized
formation_energy_ev_standardized
e_above_hull_ev_standardized
band_gap_ev_standardized
nsites_standardized
density_gm_cc_standardized
volume_standardized
e_above_hull_ev_log
band_gap_ev_log
nsites_log
density_gm_cc_log
volume_log

MIN-MAX NORMALIZATION CHECK


,min,max
formation_energy_ev_normalized,0.0,1.0
e_above_hull_ev_normalized,0.0,1.0
band_gap_ev_normalized,0.0,1.0
nsites_normalized,0.0,1.0
density_gm_cc_normalized,0.0,1.0
volume_normalized,0.0,1.0



STANDARDIZATION CHECK


,mean,std
formation_energy_ev_standardized,-1.110878e-15,1.001478
e_above_hull_ev_standardized,-2.102197e-17,1.001483
band_gap_ev_standardized,3.667993e-17,1.001478
nsites_standardized,-1.309998e-16,1.001478
density_gm_cc_standardized,-5.554390e-16,1.001478
volume_standardized,2.095996e-16,1.001478



MISSING VALUE CHECK
Total missing values: 5

TRANSFORMED DATASET SAVED
Output file: lithium_ion_transformed.csv
Final shape: (339, 30)

DATA TRANSFORMATION COMPLETED


# Conclusion

The cleaned Lithium-Ion Battery Materials dataset was successfully
transformed for further statistical and exploratory analysis.

New derived variables were created to provide additional information about
material stability and the relationship between density and volume.
Numerical variables were transformed using Min-Max normalization and
Z-score standardization.

Logarithmic transformations were also applied to appropriate non-negative
physical variables to provide alternative representations of their
distributions.

The transformed dataset was saved as `lithium_ion_transformed.csv` and is
ready for univariate, bivariate, and multivariate analysis.